In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar

RESULTS = Path("../results")

CONF = RESULTS / "week12" / "confirmatory"
HARM = RESULTS / "week12" / "harmonized_n100"
OUT = RESULTS / "week12" / "final_observed_analysis"

OUT.mkdir(parents=True, exist_ok=True)

In [3]:
# harmonized dataset
confirmatory = pd.read_csv(
    CONF / "primary_power_runs.csv",
    keep_default_na=False,
    na_values=[""]
)

n100 = pd.read_csv(
    HARM / "harmonized_n100_power_runs.csv",
    keep_default_na=False,
    na_values=[""]
)

confirmatory = confirmatory[
    confirmatory["N"].isin([50, 200])
].copy()

confirmatory["source_status"] = "confirmatory"
n100["source_status"] = "posthoc_harmonized_n100"

runs = pd.concat(
    [confirmatory, n100],
    ignore_index=True
)

assert len(runs) == 37000
assert runs["run_id"].is_unique
assert runs["reject"].isin([0, 1]).all()

print(runs["N"].value_counts().sort_index())

N
50     12000
100    15000
200    10000
Name: count, dtype: int64


In [4]:
# final observed power + Wilson CI table
power_table = (
    runs
    .groupby(
        [
            "scenario",
            "severity",
            "N",
            "test",
            "source_status"
        ],
        as_index=False
    )
    .agg(
        rejections=("reject", "sum"),
        repetitions=("reject", "size")
    )
)

power_table["power"] = (
    power_table["rejections"]
    / power_table["repetitions"]
)

In [5]:
# wilson ci
ci = power_table.apply(
    lambda row: proportion_confint(
        count=row["rejections"],
        nobs=row["repetitions"],
        alpha=0.05,
        method="wilson"
    ),
    axis=1
)

power_table["wilson_low"] = [
    x[0] for x in ci
]

power_table["wilson_high"] = [
    x[1] for x in ci
]

In [6]:
# check
assert len(power_table) == 185
assert (power_table["repetitions"] == 200).all()

power_table.head()

,scenario,severity,N,test,source_status,rejections,repetitions,power,wilson_low,wilson_high
0,Broken Timing,0.1,50,Energy,confirmatory,77,200,0.385,0.320333,0.454001
1,Broken Timing,0.1,50,MMD,confirmatory,75,200,0.375,0.310854,0.443857
2,Broken Timing,0.1,50,Sinkhorn,confirmatory,10,200,0.050,0.027383,0.089578
3,Broken Timing,0.1,100,C2ST-Linear,posthoc_harmonized_n100,14,200,0.070,0.042152,0.114055
4,Broken Timing,0.1,100,C2ST-RBF,posthoc_harmonized_n100,5,200,0.025,0.010725,0.057178


In [7]:
# save
power_table.to_csv(
    OUT / "final_observed_power_wilson.csv",
    index=False
)

In [8]:
# sanity checks 
power_table[
    (power_table["scenario"] == "Entity Inconsistency")
    & (power_table["severity"] == 0.10)
    & (power_table["N"] == 200)
][
    [
        "test",
        "rejections",
        "power",
        "wilson_low",
        "wilson_high"
    ]
]

,test,rejections,power,wilson_low,wilson_high
43,C2ST-Linear,0,0.000,0.000000,0.018845
44,C2ST-RBF,0,0.000,0.000000,0.018845
45,Energy,199,0.995,0.972226,0.999117
46,MMD,199,0.995,0.972226,0.999117
47,Sinkhorn,16,0.080,0.049841,0.125989


In [9]:
# paired contrasts function
def paired_contrast(
    df,
    scenario,
    severity,
    N,
    test_a,
    test_b
):

    a = (
        df[
            (df["scenario"] == scenario)
            & (df["severity"] == severity)
            & (df["N"] == N)
            & (df["test"] == test_a)
        ][["rep", "reject"]]
        .rename(columns={"reject": "reject_a"})
    )

    b = (
        df[
            (df["scenario"] == scenario)
            & (df["severity"] == severity)
            & (df["N"] == N)
            & (df["test"] == test_b)
        ][["rep", "reject"]]
        .rename(columns={"reject": "reject_b"})
    )

    paired = a.merge(
        b,
        on="rep",
        how="inner"
    )

    assert len(paired) == 200

    n00 = (
        (paired["reject_a"] == 0)
        & (paired["reject_b"] == 0)
    ).sum()

    n01 = (
        (paired["reject_a"] == 0)
        & (paired["reject_b"] == 1)
    ).sum()

    n10 = (
        (paired["reject_a"] == 1)
        & (paired["reject_b"] == 0)
    ).sum()

    n11 = (
        (paired["reject_a"] == 1)
        & (paired["reject_b"] == 1)
    ).sum()

    table = np.array([
        [n00, n01],
        [n10, n11]
    ])

    result = mcnemar(
        table,
        exact=True
    )

    return {
        "scenario": scenario,
        "severity": severity,
        "N": N,
        "test_a": test_a,
        "test_b": test_b,
        "both_no_reject": n00,
        "b_only_reject": n01,
        "a_only_reject": n10,
        "both_reject": n11,
        "mcnemar_statistic": result.statistic,
        "p_value": result.pvalue
    }

In [10]:
# Contrast 1 — Entity blind spot
paired_contrast(
    runs,
    "Entity Inconsistency",
    0.10,
    200,
    "C2ST-Linear",
    "MMD"
)

{'scenario': 'Entity Inconsistency',
 'severity': 0.1,
 'N': 200,
 'test_a': 'C2ST-Linear',
 'test_b': 'MMD',
 'both_no_reject': 1,
 'b_only_reject': 199,
 'a_only_reject': 0,
 'both_reject': 0,
 'mcnemar_statistic': 0.0,
 'p_value': 2.4892061111444567e-60}

- neither detects: 1
- MMD only: 199
- C2ST only: 0
- both: 0
- $p<.001$

In [11]:
# contrast 2 — RBF fixes blind spot?
paired_contrast(
    runs,
    "Entity Inconsistency",
    0.10,
    200,
    "C2ST-RBF",
    "MMD"
)

{'scenario': 'Entity Inconsistency',
 'severity': 0.1,
 'N': 200,
 'test_a': 'C2ST-RBF',
 'test_b': 'MMD',
 'both_no_reject': 1,
 'b_only_reject': 199,
 'a_only_reject': 0,
 'both_reject': 0,
 'mcnemar_statistic': 0.0,
 'p_value': 2.4892061111444567e-60}

- MMD only = 199
- RBF only = 0
- $p<.001$
- this supports the idea that simply changing from linear to RBF doesn't solve low-severity Entity blind spot.


In [12]:
# contrast 3 - over polished
paired_contrast(
    runs,
    "Over-Polished Language",
    0.10,
    200,
    "C2ST-Linear",
    "MMD"
)

{'scenario': 'Over-Polished Language',
 'severity': 0.1,
 'N': 200,
 'test_a': 'C2ST-Linear',
 'test_b': 'MMD',
 'both_no_reject': 0,
 'b_only_reject': 189,
 'a_only_reject': 0,
 'both_reject': 11,
 'mcnemar_statistic': 0.0,
 'p_value': 2.5489470578119236e-57}

- MMD only = 189
- C2ST only = 0
- both reject = 11
- $p<.001$
Shows another blind spot, in a different representation: Message MiniLM.


In [13]:
# contrast 4 - broken timing 
paired_contrast(
    runs,
    "Broken Timing",
    0.10,
    200,
    "C2ST-Linear",
    "MMD"
)

{'scenario': 'Broken Timing',
 'severity': 0.1,
 'N': 200,
 'test_a': 'C2ST-Linear',
 'test_b': 'MMD',
 'both_no_reject': 0,
 'b_only_reject': 133,
 'a_only_reject': 0,
 'both_reject': 67,
 'mcnemar_statistic': 0.0,
 'p_value': 1.8367099231598242e-40}

- MMD only = 133
- C2ST only = 0
- both reject = 67
- $p<.001$

With the structural 13D representation, the pattern C2ST vs MMD is not exclusive to a semantic embeddings phenomenon. 

In [14]:
selected_contrasts = [
    paired_contrast(
        runs,
        "Entity Inconsistency",
        0.10,
        200,
        "C2ST-Linear",
        "MMD"
    ),

    paired_contrast(
        runs,
        "Entity Inconsistency",
        0.10,
        200,
        "C2ST-RBF",
        "MMD"
    ),

    paired_contrast(
        runs,
        "Over-Polished Language",
        0.10,
        200,
        "C2ST-Linear",
        "MMD"
    ),

    paired_contrast(
        runs,
        "Broken Timing",
        0.10,
        200,
        "C2ST-Linear",
        "MMD"
    )
]

paired_table = pd.DataFrame(
    selected_contrasts
)

paired_table.to_csv(
    OUT / "selected_paired_contrasts.csv",
    index=False
)

paired_table

,scenario,severity,N,test_a,test_b,both_no_reject,b_only_reject,a_only_reject,both_reject,mcnemar_statistic,p_value
0,Entity Inconsistency,0.1,200,C2ST-Linear,MMD,1,199,0,0,0.0,2.489206e-60
1,Entity Inconsistency,0.1,200,C2ST-RBF,MMD,1,199,0,0,0.0,2.489206e-60
2,Over-Polished Language,0.1,200,C2ST-Linear,MMD,0,189,0,11,0.0,2.548947e-57
3,Broken Timing,0.1,200,C2ST-Linear,MMD,0,133,0,67,0.0,1.836710e-40
